# window_confirm

> Phase 25 (`specs/multiagent/phase25/`, experiment C1d): confirm the mirror window. On C1b's 5 exploratory boards the window (`mirrorWindow`: no bridges; each squad sees the other squad's mirror tally on every coupled clue) beat `bridge1` on all 5, with liars (+0.017) and on clean boards (+0.066). C1d tests `mirrorWindow` against `bridge1` on 8 fresh paired seeds (1820-1827), with liars (the gate) and clean (a secondary test). `ARMS25`, `c1dGrid` and `c1dRange`; `C1D_PLAN`, frozen after the user approved the design and its $20.35-21.37 estimate; `gateC1d` (part i, effective; part ii, safe), `cleanC1d`, `c1dSummary` and `writeC1d`.

In [ ]:
#| default_exp window_confirm

In [ ]:
#| hide
from nbdev.showdoc import *

## Grid and cost

Two arms from `ARMS23`, unchanged: C1's `bridge1` (a solver bridge per squad pair) and C1b's `mirrorWindow` (no bridges; the receiving squad's board shows every coupled clue's mirror tally from the start, FR-A3 of Phase 23). 64 agents, seeds 1820-1827 (fresh: C1 and C1b used 1800-1807, C1c 1810-1817, the pilots 1890). Each arm on each seed with independent liars (the gate) and clean (no liars, the secondary test). The cost per board is the measured mean over C1's and C1b's boards of that arm and fault; the high end allows 5%.

In [ ]:
#| export
from explRL.bridge_report import _d, _test
from explRL.clue_colony import checkOptions
from explRL.colony_grid import WORLD18, colonyCell
from explRL.colony_report import bootSeeds, cellVals
from explRL.relay_grid import ARMS23, N20
from explRL.role_confirm import gateC1c

ARMS25 = dict(bridge1=ARMS23['bridge1'], mirrorWindow=ARMS23['mirrorWindow'])
SEEDS25 = list(range(1820, 1828))                                   # fresh seeds, paired across the two arms and both faults
FAULTS25 = ('independent', 'clean')
COST25 = dict(independent=dict(bridge1=0.651, mirrorWindow=0.686),  # $ per board: C1's and C1b's boards of each arm and fault
              clean=dict(bridge1=0.609, mirrorWindow=0.598))
ALLOW25 = 1.05


def c1dCell(arm, seed, fault='independent', runner='llm'):
    "One C1d cell: study colony25, level '<arm>|<fault>', N 64, checked."
    c = colonyCell('colony25', arm, N20, seed, fault, runner, arms=ARMS25)
    checkOptions(c['cfg']['world'], N20, c['cfg'].get('nLiar', 0))
    return c


def c1dGrid(runner='llm'):
    "The 32 boards: each arm on SEEDS25, with independent liars and clean."
    return [c1dCell(a, s, f, runner) for f in FAULTS25 for a in ARMS25 for s in SEEDS25]


def c1dRange(per=COST25, allow=ALLOW25):
    "(low, high) $ for c1dGrid: the measured cost per board by fault and arm, and that times the allowance."
    low = sum(per[c['level'].split('|')[1]][c['level'].split('|')[0]] for c in c1dGrid())
    return round(low, 2), round(low * allow, 2)

In [ ]:
# test_c1d_grid (FR-1, FR-3): 32 boards, fresh seeds, both faults, the two arms' options unchanged from C1b, and the estimate
g = c1dGrid()
assert len(g) == 32 and {c['level'] for c in g} == {f'{a}|{f}' for a in ARMS25 for f in FAULTS25}
assert sorted({c['seed'] for c in g}) == SEEDS25 and not set(SEEDS25) & (set(range(1800, 1818)) | {1890})
assert all(c['N'] == 64 for c in g)
assert {c['cfg'].get('nLiar', 0) for c in g if c['level'].endswith('independent')} == {16}
assert {c['cfg'].get('nLiar', 0) for c in g if c['level'].endswith('clean')} == {0}
w = {c['level'].split('|')[0]: c['cfg']['world'] for c in g}
assert w['mirrorWindow'].get('mirrorWindow') and not w['mirrorWindow'].get('bridges') and w['bridge1'].get('bridges') == 1
assert not w['bridge1'].get('mirrorWindow') and c1dRange() == (20.35, 21.37), c1dRange()

## The plan

`C1D_PLAN` holds the gate, what it decides for A1, the clean-board secondary test, the calls recorded by hand before the run (as P23-10: pilot-only from C1b's 5 boards, and every-design-helps) and the cost rule. It was frozen on 2026-11-12, after the user approved the design and the estimate; its hash is pinned below.

In [ ]:
#| export
_G = 'mirrorWindow - bridge1, N 64, independent liars, seeds 1820-1827 paired'
C1D_PLAN = dict(
    version=1, frozen=True, plan='specs/multiagent/phase25/', model='claude-haiku-5-5', world=WORLD18, turns=40,   # frozen 2026-11-12 (the user approved the design and the $20.35-21.37 estimate)
    channel='blackboard', memory='full', effort='low', arms=ARMS25,
    cells=dict(liars={a: SEEDS25 for a in ARMS25}, clean={a: SEEDS25 for a in ARMS25}), seeds=SEEDS25,
    fault=dict(independent='scripted ClueLiar in the first 16 slots (MirrorLiar: also on mirrors), guards the next 16',
               clean='no liars: 64 Claude agents, the 16 guards in the first 16 slots'),
    hypotheses=dict(
        H25_1=f'Gate C1d (i), effective: honestPooled, {_G}: one-sided exact sign-flip > 0, alpha 0.05 (smallest p 1/256)',
        H25_2=f'Gate C1d (ii), safe (non-inferiority): contamination, {_G}: the 95th percentile of 2,000 paired bootstrap means < 0.05',
        H25_3='secondary, not in the gate: honestPooled, mirrorWindow - bridge1, N 64, clean boards, seeds 1820-1827 paired: one-sided '
              'exact sign-flip > 0, alpha 0.05, uncorrected; welfareHonest beside it'),
    gate=dict(parts=['H25_1', 'H25_2'], arm='mirrorWindow', base='bridge1', alpha=0.05, metric='honestPooled', safety='contamination',
              margin=0.05, quantile=95, decides=dict(
                  passes="A1's bridged world replaces bridges with the mirror window (amends P19-8): no bridge agents, so relay credit "
                         "(P20-12) lapses and the 3B check (P20-11) measures the window's prompt length; the border contest credit "
                         "(P23-8) stays",
                  fails='A1 keeps bridge1 (P19-8 as settled by C1b and C1c); the window is not adopted')),
    robustness='part (i) on welfareHonest; an estimate of the opposite sign marks it not robust',
    why="Gates C1b and C1c (2026-11-12) left A1 on bridge1, which relays on under 1% of its turns; on C1b's 5 exploratory boards "
        "the mirror window beat bridge1 on all 5, with liars (+0.017) and clean (+0.066), at no extra agent; it was chosen after "
        "seeing them, so the user asked for this confirmation",
    calls=dict(basis="C1b's 5 paired boards, seeds 1800-1804 (exploratory): mirrorWindow beat bridge1 on all 5, with liars and clean",
               pilotOnly=dict(H25_1=dict(call='passes', estimate=0.0174), H25_2=dict(call='holds', estimate=0.0165),
                              H25_3=dict(call='supported', estimate=0.0658), gate='passes'),
               everyDesignHelps=dict(H25_1=dict(call='passes'), H25_2=dict(call='holds'), H25_3=dict(call='supported'), gate='passes')),
    noPilot='both arms ran at this size, with liars and clean, in C1 and C1b with invalid replies under 0.2% (the user, 2026-11-12)',
    costRule=dict(rule='report and wait (P20-8): the user approved the estimate before the run', ceiling=23.0,
                  estimate=c1dRange(), perBoard=COST25, allowance=ALLOW25),
    stop='invalid replies >= 5% or the monthly cap: stop and ask', approved=dict(design='2026-11-12', estimate=c1dRange()),
    cap=250.0, capNote='per calendar month (the user, 2026-11-09)', dirs=dict(main='results/multiagent/colony/c1d'),
    nboot=2000, confirmatory=True)

In [ ]:
# test_c1d_plan (FR-2): frozen with its hash pinned; the gate, the secondary test, the calls and the approval recorded
from explRL.coop_grid import coopHash
assert C1D_PLAN['frozen'] and coopHash(C1D_PLAN) == '27ff916003d1', coopHash(C1D_PLAN)
assert C1D_PLAN['cap'] == 250.0 and 'calendar month' in C1D_PLAN['capNote'] and C1D_PLAN['gate']['margin'] == 0.05
assert set(C1D_PLAN['hypotheses']) == {'H25_1', 'H25_2', 'H25_3'} and set(C1D_PLAN['gate']['decides']) == {'passes', 'fails'}
assert all(set(C1D_PLAN['hypotheses']) <= set(C1D_PLAN['calls'][k]) for k in ('pilotOnly', 'everyDesignHelps'))
assert C1D_PLAN['approved']['estimate'] == c1dRange() and c1dRange()[1] <= C1D_PLAN['costRule']['ceiling']

## The gate and the report

The gate is `role_confirm.gateC1c` read with `C1D_PLAN` (arm `mirrorWindow`, base `bridge1`; it uses only the boards with liars). `cleanC1d` is H25-3 on the clean boards.

In [ ]:
#| export
def gateC1d(rows, plan=C1D_PLAN, nboot=2000, metric='honestPooled'):
    "Gate C1d: `gateC1c` with C1D_PLAN's arm, base, test and margin, on the boards with independent liars."
    return gateC1c(rows, plan, nboot, metric)


def cleanC1d(rows, plan=C1D_PLAN, nboot=2000):
    "H25-3: arm - base on the clean boards, honestPooled (one-sided exact sign-flip, supported at p < alpha with a positive estimate) and welfareHonest."
    G, s8 = plan['gate'], set(plan['seeds'])
    out = {m: _test(_d(rows, G['arm'], G['base'], m, fault='clean', seeds=s8), nboot) for m in ('honestPooled', 'welfareHonest')}
    out['supported'] = bool(out['honestPooled']['p'] < G['alpha'] and out['honestPooled']['estimate'] > 0)
    return out


C1D_MEASURES = ('honestPooled', 'welfareHonest', 'contamination', 'coupledContamination', 'crossContamination', 'sharePerHeld', 'inTokPerCall')


def c1dSummary(rows, plan=C1D_PLAN, nboot=None):
    """The gate with part (i)'s robustness on welfareHonest, H25-3 on the clean boards, each arm's measures per fault (mean and
    bootstrap interval over its boards), and the hand-recorded calls scored. ValueError unless all 32 planned cells are present."""
    nboot = nboot or plan['nboot']
    want = {(f'{a}|{f}', s) for f, k in (('independent', 'liars'), ('clean', 'clean')) for a, ss in plan['cells'][k].items() for s in ss}
    key = lambda r: (r['level'], int(float(r['seed'])))
    rows = [r for r in rows if int(float(r['N'])) == N20 and key(r) in want]
    if miss := sorted(want - {key(r) for r in rows}):
        raise ValueError(f'{len(miss)} planned C1d cells missing, e.g. {miss[:4]}')
    gate, rob = gateC1d(rows, plan, nboot), gateC1d(rows, plan, nboot, 'welfareHonest')
    gate['robust'], gate['welfareHonest'] = bool(rob['i']['estimate'] * gate['i']['estimate'] > 0), rob['i']
    clean = cleanC1d(rows, plan, nboot)
    arms = {f: {a: {m: bootSeeds({k: v for k, v in cellVals(rows, a, f, m, [N20]).items() if v == v}, nboot=nboot)
                    for m in C1D_MEASURES} for a in plan['arms']} for f in FAULTS25}
    got = dict(H25_1='passes' if gate['i']['passes'] else 'fails', H25_2='holds' if gate['ii']['holds'] else 'fails',
               H25_3='supported' if clean['supported'] else 'not supported', gate=gate['verdict'])
    calls = {k: {h: dict(call=v['call'] if isinstance(v, dict) else v, observed=got[h],
                         right=(v['call'] if isinstance(v, dict) else v) == got[h]) for h, v in plan['calls'][k].items()}
             for k in ('pilotOnly', 'everyDesignHelps')}
    cost = sum(float(r.get('cost', 0) or 0) for r in rows)
    return dict(gate=gate, clean=clean, arms=arms, calls=calls, boards=len(rows), cost=cost,
                invalid=sum(float(r['invalid']) for r in rows) / (sum(float(r['calls']) for r in rows) or 1))


def writeC1d(path, S, meta, plan=C1D_PLAN):
    "report.md: the gate, part (i)'s robustness, H25-3, the arms side by side per fault and the calls."
    g, c, f = S['gate'], S['clean'], lambda x: 'n/a' if x != x else f'{x:.3f}'
    ci = lambda t: f"[{f(t['ci'][0])}, {f(t['ci'][1])}]"
    row = lambda name, t, v: f"| {name} | {t['n']} | {f(t['estimate'])} | {ci(t)} | {t['p']:.3f} | {v} |"
    L = ['# Phase 25 (C1d): confirm the mirror window (confirmatory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, boards {S['boards']}, cost ${S['cost']:.2f}, invalid {S['invalid']:.4f}", '',
         f"## Gate C1d ({plan['gate']['arm']} - {plan['gate']['base']}, 64 agents, independent liars, seeds 1820-1827 paired)", '',
         '| item | boards | estimate | 95% interval | p (one-sided, exact) | verdict |', '|---|---|---|---|---|---|',
         row('(i) effective: honestPooled', g['i'], f"passes: {g['i']['passes']}"),
         row('(i) robustness: welfareHonest', g['welfareHonest'], f"same sign: {g['robust']}"),
         row('(ii) safe: contamination (p for < 0)', g['ii'], f"holds: {g['ii']['holds']} (95th percentile {f(g['ii']['upper'])} against {g['ii']['margin']})"),
         '', f"**Gate C1d: {g['verdict']}.** {g['decides']}", '',
         '## H25-3: clean boards (secondary, uncorrected, not in the gate)', '',
         '| item | boards | estimate | 95% interval | p (one-sided, exact) | verdict |', '|---|---|---|---|---|---|',
         row('honestPooled', c['honestPooled'], f"supported: {c['supported']}"), row('welfareHonest', c['welfareHonest'], '')]
    for fl, A in S['arms'].items():
        L += ['', f'## Per arm, {fl} (mean [95% interval] over 8 boards)', '',
              '| arm | ' + ' | '.join(C1D_MEASURES) + ' |', '|---|' + '---|' * len(C1D_MEASURES)]
        L += [f'| {a} | ' + ' | '.join(f'{f(v[0])} [{f(v[1])}, {f(v[2])}]' for v in A[a].values()) + ' |' for a in A]
    L += ['', '## Calls recorded before the run', '', '| line | item | call | observed | right |', '|---|---|---|---|---|']
    L += [f'| {k} | {h} | {c2["call"]} | {c2["observed"]} | {c2["right"]} |' for k, cs in S['calls'].items() for h, c2 in cs.items()]
    open(path, 'w').write('\n'.join(L) + '\n')
    return '\n'.join(L)

In [ ]:
# test_c1d_gate (FR-4, FR-5): synthetic rows: a clear gain that is safe passes; no gain fails; a gain that is less safe fails;
# the clean test reads only clean boards; missing cells refuse; the report writes the verdict and scores the calls
import tempfile
from pathlib import Path


def rows25(gain, dcont, cleanGain=0.06, noise=0.004):
    out = []
    for fl in FAULTS25:
        for a in ARMS25:
            for i, s in enumerate(SEEDS25):
                e, w = (i % 3 - 1) * noise, a == 'mirrorWindow'
                hp = (0.18 if fl == 'independent' else 0.28) + e + (w * (gain if fl == 'independent' else cleanGain))
                out.append(dict(level=f'{a}|{fl}', seed=s, N=64, honestPooled=hp, welfareHonest=hp + 0.05, calls=1920.0, invalid=1.0,
                                contamination=(0.33 + e + w * dcont) if fl == 'independent' else 0.0, coupledContamination=0.7,
                                crossContamination=0.0, sharePerHeld=0.5, inTokPerCall=900.0, cost=0.65))
    return out


S = c1dSummary(rows25(0.02, 0.01), nboot=200)
assert S['gate']['verdict'] == 'passes' and S['gate']['i']['p'] == 1 / 256 and S['gate']['robust'] and S['boards'] == 32
assert S['clean']['supported'] and abs(S['clean']['honestPooled']['estimate'] - 0.06) < 1e-9
assert all(c['right'] for cs in S['calls'].values() for c in cs.values()) and abs(S['cost'] - 32 * 0.65) < 1e-9
assert gateC1d(rows25(0.0, 0.0), nboot=200)['verdict'] == 'fails'                        # no gain
G = gateC1d(rows25(0.02, 0.08), nboot=200)
assert G['i']['passes'] and not G['ii']['holds'] and G['verdict'] == 'fails'            # effective but less safe
assert gateC1d(rows25(-0.02, 0.0), nboot=200)['verdict'] == 'fails'                      # worse
S0 = c1dSummary(rows25(0.02, 0.01, cleanGain=0.0), nboot=200)                            # the clean test does not move the gate
assert S0['gate']['verdict'] == 'passes' and not S0['clean']['supported'] and not S0['calls']['pilotOnly']['H25_3']['right']
try:
    c1dSummary(rows25(0.02, 0.01)[1:]); assert False
except ValueError:
    pass
p = Path(tempfile.mkdtemp()) / 'report.md'
txt = writeC1d(p, S, dict(planHash='x', commit='y'))
assert 'Gate C1d: passes.' in txt and 'mirrorWindow' in txt and '## H25-3' in txt and p.read_text() == txt + '\n'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()